<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 2 — Nettoyage et Validation de la Base `ressources.csv`

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif de cette phase

La base `ressources.csv` est le **journal de présence quotidienne** des agents.  
Chaque ligne correspond à une journée de présence d'un agent avec ses caractéristiques contractuelles.

Ce notebook applique toutes les corrections décrites dans [`docs/phase2_ressources.md`](../docs/phase2_ressources.md).

**Démarche pour chaque anomalie :**
1. **Constat** — quantification du problème
2. **Décision** — avec justification métier/statistique
3. **Action** — code de correction
4. **Vérification** — preuve que la correction est effective


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- Localisation de la racine du projet
def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine (présence de .git ou requirements.txt)."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))

from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
# --- Chargement de la base ressources
ressources_original = pd.read_csv('data/ressources.csv', encoding='latin-1')

# Copie de travail — on ne touche JAMAIS à ressources_original
ressources = ressources_original.copy()

print(f" Dimensions : {ressources.shape[0]:,} lignes × {ressources.shape[1]} colonnes")
print(f" Agents uniques (Matricule) : {ressources['Matricule'].nunique():,}")
print()
print("Types de données :")
print(ressources.dtypes)

---
## Section 1 — Vue d'ensemble de la qualité des données

### 1.1 Valeurs uniques par colonne catégorielle

In [ ]:
colonnes_categorielles = ['Lieu.travail', 'Population', 'Site', 'Type.de.contrat']

for colonne in colonnes_categorielles:
    valeurs_uniques = ressources[colonne].value_counts()
    print(f"\n{colonne} ({ressources[colonne].nunique()} valeurs uniques) :")
    print(valeurs_uniques.to_string())

### 1.2 Tableau récapitulatif des anomalies

In [ ]:
def compter_anomalies_par_colonne(dataframe):
    """Calcule, pour chaque colonne, le nombre de NaN, de '???', et le total."""
    resultats = []
    for colonne in dataframe.columns:
        n_nan           = dataframe[colonne].isna().sum()
        n_interrogation = (dataframe[colonne].astype(str).str.strip() == '???').sum()
        total           = n_nan + n_interrogation
        pourcentage     = round(total / len(dataframe) * 100, 2)
        resultats.append({
            'Colonne'        : colonne,
            'NaN'            : n_nan,
            '???'            : n_interrogation,
            'Total anomalies': total,
            '% du total'     : pourcentage
        })
    return (pd.DataFrame(resultats)
              .sort_values('Total anomalies', ascending=False)
              .reset_index(drop=True))

tableau_anomalies_initial = compter_anomalies_par_colonne(ressources)
print("Tableau des anomalies initiales :")
tableau_anomalies_initial

---
## Section 2 — Anomalie B1 : Normalisation du format de `Date.presence`

### Constat
Toutes les dates sont au format français `DD/MM/YYYY`.

### Décision : Conversion vers le format ISO `YYYY/MM/DD`
> Cohérence avec les bases `dossier` et `temps` (format ISO).  
> Facilite les jointures temporelles et les comparaisons chronologiques sans conversion à la volée.

In [ ]:
# Aperçu du format actuel
print("Exemples de dates avant conversion :")
print(ressources['Date.presence'].head(5).tolist())
print()

def convertir_date_vers_iso(valeur_date_en_chaine):
    """Convertit DD/MM/YYYY vers YYYY/MM/DD. Retourne la valeur originale en cas d'échec."""
    try:
        return pd.to_datetime(valeur_date_en_chaine, dayfirst=True).strftime('%Y/%m/%d')
    except Exception:
        return valeur_date_en_chaine

valeurs_avant_conversion  = ressources['Date.presence'].copy()
ressources['Date.presence'] = ressources['Date.presence'].apply(convertir_date_vers_iso)
masque_dates_modifiees    = valeurs_avant_conversion != ressources['Date.presence']

print(f" {masque_dates_modifiees.sum():,} date(s) convertie(s) au format ISO")
print()
print("Exemples après conversion :")
print(ressources['Date.presence'].head(5).tolist())

In [ ]:
# Vérification — toutes les dates doivent correspondre au format YYYY/MM/DD
masque_format_non_iso = ~ressources['Date.presence'].str.fullmatch(r'\d{4}/\d{2}/\d{2}', na=False)
n_non_iso = masque_format_non_iso.sum()

print(f" Vérification : {n_non_iso} date(s) non-conforme(s) au format ISO restante(s)")
if n_non_iso > 0:
    print("Exemples non conformes :")
    ressources[masque_format_non_iso]['Date.presence'].head(5)
else:
    print(" Toutes les dates sont au format YYYY/MM/DD")

---
## Section 3 — Anomalie B2 : Analyse de la plage temporelle

### Constat
La base `dossier` couvre exclusivement 2021–2022.  
La base `ressources` peut contenir des présences hors de cette fenêtre.

### Décision : Signalement sans suppression
> `ressources` est une **table de référence RH** — sa portée temporelle peut dépasser celle des dossiers.  
> Les présences hors-période seront simplement exclues lors de la jointure en Phase 4.  
> Supprimer des agents entiers réduirait inutilement la population de référence.

In [ ]:
ressources['annee_presence'] = ressources['Date.presence'].str[:4]
distribution_annees = ressources['annee_presence'].value_counts().sort_index()

print("Distribution des présences par année :")
print(distribution_annees.to_frame())
print()

# Visualisation
fig, ax = plt.subplots(figsize=(10, 4))
couleurs = ['#c00000' if annee not in ['2021', '2022'] else '#4472c4'
            for annee in distribution_annees.index]
ax.bar(distribution_annees.index, distribution_annees.values, color=couleurs)
ax.set_title("Distribution des présences par année (rouge = hors période 2021-2022)", fontsize=12)
ax.set_xlabel("Année")
ax.set_ylabel("Nombre de lignes")
for i, (annee, n) in enumerate(distribution_annees.items()):
    ax.text(i, n + 500, f"{n:,}", ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('data/phase2_distribution_annees.png', dpi=150, bbox_inches='tight')
plt.show()

n_hors_periode = distribution_annees[~distribution_annees.index.isin(['2021', '2022'])].sum()
print(f"  {n_hors_periode:,} ligne(s) hors période 2021–2022 → conservées dans ressources, filtrées en Phase 4")

---
## Section 4 — Anomalies B3 & B4 : Valeurs manquantes et `???`

### Constat B3
0 NaN dans toutes les colonnes — la base est complète.

### Décision B3 : Aucune action
> La complétude est vérifiée programmatiquement ci-dessous et documentée pour traçabilité.

In [ ]:
# --- Vérification des NaN
bilan_nan = ressources.isna().sum()
print("Valeurs NaN par colonne :")
print(bilan_nan.to_frame(name='NaN'))
print()

if bilan_nan.sum() == 0:
    print(" Aucune valeur manquante (NaN) dans la base ressources")
else:
    print(f"  {bilan_nan.sum():,} valeur(s) manquante(s) détectée(s)")

In [ ]:
# --- Vérification des '???'
bilan_interrogation = {}
for colonne in ressources.columns:
    n = (ressources[colonne].astype(str).str.strip() == '???').sum()
    bilan_interrogation[colonne] = n

serie_interrogation = pd.Series(bilan_interrogation)
print("Valeurs '???' par colonne :")
print(serie_interrogation.to_frame(name="???"))
print()

if serie_interrogation.sum() == 0:
    print(" Aucune valeur '???' dans la base ressources")
else:
    colonnes_avec_qqq = serie_interrogation[serie_interrogation > 0].index.tolist()
    for colonne in colonnes_avec_qqq:
        ressources[colonne] = ressources[colonne].replace('???', np.nan)
        print(f" {colonne} : {bilan_interrogation[colonne]:,} '???' → NaN")

---
## Section 5 — Anomalie B5 : Analyse des outliers dans `Duree.travail`

### Constat
`Duree.travail` représente les heures travaillées dans la journée.  
Statistiques : min=0.017h, max=23.42h, moyenne=6.15h, médiane=7.33h.

### Décision : Signalement avec visualisation — conservation sans suppression
> Une durée de 1 minute peut être une errée de badgeage ou un départ anticipé légitime.  
> Une durée de 23h peut correspondre à une garde de nuit.  
> Sans règle métier précise, la suppression serait arbitraire.  
> Ces cas seront traités comme outliers dans les modèles (Phase 5 & 6).

In [ ]:
print("Statistiques de Duree.travail :")
print(ressources['Duree.travail'].describe())
print()

# Seuils d'alerte : < 1h ou > 12h (journée standard ≤ 12h)
SEUIL_MIN_HEURES = 1.0
SEUIL_MAX_HEURES = 12.0

masque_duree_tres_courte  = ressources['Duree.travail'] < SEUIL_MIN_HEURES
masque_duree_tres_longue  = ressources['Duree.travail'] > SEUIL_MAX_HEURES

print(f"Présences < {SEUIL_MIN_HEURES}h  : {masque_duree_tres_courte.sum():,} ({masque_duree_tres_courte.mean()*100:.2f}%)")
print(f"Présences > {SEUIL_MAX_HEURES}h : {masque_duree_tres_longue.sum():,} ({masque_duree_tres_longue.mean()*100:.2f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Distribution de Duree.travail", fontsize=13, fontweight='bold')

axes[0].hist(ressources['Duree.travail'], bins=50, color='#4472c4', edgecolor='white')
axes[0].axvline(SEUIL_MIN_HEURES, color='red', linestyle='--', label=f'< {SEUIL_MIN_HEURES}h')
axes[0].axvline(SEUIL_MAX_HEURES, color='orange', linestyle='--', label=f'> {SEUIL_MAX_HEURES}h')
axes[0].set_title("Distribution complète")
axes[0].set_xlabel("Heures travaillées")
axes[0].set_ylabel("Nombre de lignes")
axes[0].legend()

# Boxplot par Lieu.travail
ressources.boxplot(column='Duree.travail', by='Lieu.travail', ax=axes[1])
axes[1].set_title("Boxplot par lieu de travail")
axes[1].set_xlabel("Lieu de travail")
axes[1].set_ylabel("Heures travaillées")
plt.suptitle("")
plt.tight_layout()
plt.savefig('data/phase2_duree_travail.png', dpi=150, bbox_inches='tight')
plt.show()

print("→ Outliers signalés — conservation sans suppression (décision B5)")

---
## Section 6 — Anomalie B6 : Vérification de `Temps.travail`

### Constat
`Temps.travail` devrait représenter le pourcentage de temps de travail contractuel.

### Décision : Vérification — signalement si constante
> Une variable constante n'apporte aucune information discriminante aux modèles.  
> Elle sera exclue de la modélisation si sa valeur est unique.

In [ ]:
valeurs_uniques_temps_travail = ressources['Temps.travail'].value_counts()
print("Distribution de Temps.travail :")
print(valeurs_uniques_temps_travail.to_frame())
print()

n_valeurs_distinctes = ressources['Temps.travail'].nunique()
if n_valeurs_distinctes == 1:
    valeur_unique = ressources['Temps.travail'].iloc[0]
    print(f"  Temps.travail est CONSTANT : {valeur_unique} pour tous les enregistrements")
    print("→ Variable exclue de la modélisation (aucune variance)")
else:
    print(f" Temps.travail a {n_valeurs_distinctes} valeurs distinctes — variable conservée")

---
## Section 7 — Anomalie B7 : Doublons `Matricule + Date.presence`

### Constat
Un agent ne devrait avoir qu'une seule ligne de présence par jour.

### Décision : Suppression si présents — `keep='first'`
> Un double enregistrement de présence fausserait le calcul du temps travaillé,  
> variable clé pour les analyses futures (charge de travail, productivité).

In [ ]:
masque_doublons_agent_jour = ressources.duplicated(subset=['Matricule', 'Date.presence'], keep=False)
nombre_doublons_agent_jour = masque_doublons_agent_jour.sum()

print(f"Lignes avec doublon (Matricule + Date.presence) : {nombre_doublons_agent_jour:,}")

if nombre_doublons_agent_jour > 0:
    doublons_agent_jour = ressources[masque_doublons_agent_jour].sort_values(['Matricule', 'Date.presence'])
    print()
    print("Aperçu des doublons :")
    doublons_agent_jour.head(10).style_duplicates(id_column='Matricule')
else:
    print(" Aucun doublon (Matricule + Date.presence) — unicité confirmée")

In [ ]:
if nombre_doublons_agent_jour > 0:
    nombre_avant_deduplication = len(ressources)
    ressources.drop_duplicates(subset=['Matricule', 'Date.presence'], keep='first', inplace=True)
    ressources.reset_index(drop=True, inplace=True)
    nombre_supprime = nombre_avant_deduplication - len(ressources)
    print(f" {nombre_supprime:,} doublon(s) supprimé(s)")
    assert ressources.duplicated(subset=['Matricule', 'Date.presence']).sum() == 0
    print(" Vérification : 0 doublon restant ")
else:
    print(" Aucune action requise — pas de doublons")

---
## Section 8 — Anomalie B8 : Cohérence de `Experience` par agent

### Constat
`Experience` représente les jours cumulés d'expérience.  
Sa valeur doit **croître de façon monotone** pour chaque agent au fil du temps.

### Décision : Signalement des incohérences
> Une expérience décroissante est physiquement impossible.  
> Signalée pour investigation — pas de suppression car l'agent reste exploitable.

In [ ]:
# Vérification de la monotonie croissante de l'expérience par agent
def verifier_experience_croissante(groupe_agent):
    """Vérifie que l'expérience est monotone croissante pour un agent donné."""
    experience_triee = groupe_agent.sort_values('Date.presence')['Experience']
    return experience_triee.is_monotonic_increasing

resultats_experience = (
    ressources.groupby('Matricule')
    .apply(verifier_experience_croissante, include_groups=False)
)

n_agents_incoherents = (~resultats_experience).sum()
print(f"Agents avec expérience NON-croissante : {n_agents_incoherents:,} / {len(resultats_experience):,}")

if n_agents_incoherents > 0:
    matricules_incoherents = resultats_experience[~resultats_experience].index.tolist()
    print(f"Matricule(s) concerné(s) : {matricules_incoherents[:10]}")
    print("→ Signalé pour investigation — agents conservés")
else:
    print(" Expérience croissante confirmée pour tous les agents")

---
## Section 9 — Anomalie B9 : Cross-référence Matricule ↔ dossier

### Constat
Vérification croisée : tous les matricules de `dossier` doivent exister dans `ressources`.

### Décision : Confirmation du signalement de Phase 1
> Vue complémentaire du problème détecté en Phase 1 (1 matricule orphelin dans dossier).

In [ ]:
# Chargement du dossier nettoyé pour cross-référence
dossier_nettoye = pd.read_csv('data/dossier_nettoye.csv', encoding='utf-8')

ensemble_matricules_ressources = set(ressources['Matricule'].dropna().astype(int))
ensemble_matricules_dossier    = set(dossier_nettoye['Matricule.de.traitement'].dropna().astype(int))

# Matricules dans dossier mais absents de ressources
matricules_dossier_sans_ressources = ensemble_matricules_dossier - ensemble_matricules_ressources

# Matricules dans ressources mais absents de dossier
matricules_ressources_sans_dossier = ensemble_matricules_ressources - ensemble_matricules_dossier

print(f"Matricules dans ressources : {len(ensemble_matricules_ressources):,}")
print(f"Matricules dans dossier    : {len(ensemble_matricules_dossier):,}")
print()
print(f"  Dans dossier mais absent de ressources : {len(matricules_dossier_sans_ressources)}")
if matricules_dossier_sans_ressources:
    print(f"   Matricule(s) : {sorted(matricules_dossier_sans_ressources)}")
print()
print(f"ℹ  Dans ressources mais jamais dans dossier : {len(matricules_ressources_sans_dossier):,}")
print("   (Agents présents mais n'ayant traité aucun dossier dans la période)")

---
## Section 10 — Synthèse et tableau de bord de qualité

In [ ]:
bilan_transformations = pd.DataFrame([
    {'N°': 'B1', 'Anomalie': 'Format date DD/MM/YYYY',        'N initial': len(ressources_original), 'Action': 'Conversion ISO YYYY/MM/DD'},
    {'N°': 'B2', 'Anomalie': 'Dates hors période 2021–2022',  'N initial': '?', 'Action': 'Signalé — filtrage en Phase 4'},
    {'N°': 'B3', 'Anomalie': 'Valeurs NaN',                   'N initial': 0,   'Action': 'Aucune action (0 NaN)'},
    {'N°': 'B4', 'Anomalie': "Valeurs '???'",                  'N initial': '?', 'Action': 'Remplacement par NaN si présents'},
    {'N°': 'B5', 'Anomalie': 'Outliers Duree.travail',        'N initial': '?', 'Action': 'Signalé — traitement en Phase 5&6'},
    {'N°': 'B6', 'Anomalie': 'Temps.travail constant',        'N initial': '?', 'Action': 'Vérification — exclusion si constante'},
    {'N°': 'B7', 'Anomalie': 'Doublons Matricule+Date',       'N initial': '?', 'Action': 'Suppression si présents'},
    {'N°': 'B8', 'Anomalie': 'Experience non-croissante',     'N initial': '?', 'Action': 'Signalé — conservation'},
    {'N°': 'B9', 'Anomalie': 'Matricules orphelins (dossier)','N initial': 1,   'Action': 'Confirmé depuis Phase 1'},
])
print("Bilan des transformations Phase 2 :")
bilan_transformations

In [ ]:
print(f"=== ÉTAT FINAL DE LA BASE ressources ===")
print(f"Lignes  : {len(ressources):,}")
print(f"Colonnes: {ressources.shape[1]}")
print()
print("Valeurs manquantes restantes :")
manquants = ressources.isna().sum()
if manquants.sum() == 0:
    print("  → Aucune (base complète)")
else:
    print(manquants[manquants > 0])
print()
print("Aperçu des 5 premières lignes :")
ressources.head(5).style_duplicates()

---
## Section 11 — Export de la base nettoyée

In [ ]:
# Suppression de la colonne temporaire
ressources.drop(columns=['annee_presence'], errors='ignore', inplace=True)

CHEMIN_SORTIE = 'data/ressources_nettoyees.csv'
ressources.to_csv(CHEMIN_SORTIE, index=False, encoding='utf-8')

print(f" Base nettoyée exportée : {CHEMIN_SORTIE}")
print(f"   {ressources.shape[0]:,} lignes × {ressources.shape[1]} colonnes")
print()
print("Aperçu final :")
ressources.head(3).style_duplicates()